# Quantum Oracles

A **quantum oracle** is a reversible black-box operation that encodes a
classical function $f$ into a quantum circuit.  It acts on a two-register
state $|x\rangle|y\rangle$ and produces:

$$U_f\,|x\rangle|y\rangle = |x\rangle\,|y \oplus f(x)\rangle$$

where $\oplus$ denotes bitwise XOR (addition mod 2).

## Circuit diagram

```
         +-----------+
  |x> ---|           |--- |x>
         |    U_f    |
  |y> ---|           |--- |y XOR f(x)>
         +-----------+
```

The **query register** $|x\rangle$ is unchanged; the **ancilla register**
$|y\rangle$ absorbs the result.

## Why XOR?

XOR keeps the operation **reversible**: applying $U_f$ twice returns the
original state, because
$(y \oplus f(x)) \oplus f(x) = y$.
Reversibility is a hard requirement for quantum circuits.

## Phase-kickback trick

When the ancilla is set to $|{-}\rangle = (|0\rangle - |1\rangle)/\sqrt{2}$
instead of $|0\rangle$, the XOR flips become **phase flips**:

$$U_f\,|x\rangle|{-}\rangle = (-1)^{f(x)}\,|x\rangle|{-}\rangle$$

The ancilla is **unchanged**; the phase $(-1)^{f(x)}$ is kicked back onto
the query register.  This is the key mechanism behind Deutsch,
Deutsch-Jozsa, Bernstein-Vazirani, and Simon's algorithm.

## Numeric examples

**1-qubit query, $f(x) = 0$ (constant zero)**

- $|0\rangle|0\rangle \xrightarrow{U_{f_0}} |0\rangle|0 \oplus 0\rangle = |0\rangle|0\rangle$
- $|1\rangle|0\rangle \xrightarrow{U_{f_0}} |1\rangle|0 \oplus 0\rangle = |1\rangle|0\rangle$

**1-qubit query, $f(x) = 1$ (constant one)**

- $|0\rangle|0\rangle \xrightarrow{U_{f_1}} |0\rangle|0 \oplus 1\rangle = |0\rangle|1\rangle$
- $|1\rangle|0\rangle \xrightarrow{U_{f_1}} |1\rangle|0 \oplus 1\rangle = |1\rangle|1\rangle$

**Phase-kickback with $|y\rangle = |{-}\rangle$ and $f(x) = 1$**

- $|0\rangle \xrightarrow{U_{f_1}} (-1)^1\,|0\rangle|{-}\rangle = -|0\rangle|{-}\rangle$
- $|1\rangle \xrightarrow{U_{f_1}} (-1)^1\,|1\rangle|{-}\rangle = -|1\rangle|{-}\rangle$

## Imports

In [1]:
from sympy.physics.quantum.qapply import qapply
from sympy.physics.quantum.qubit import Qubit, matrix_to_qubit
from sympy import sqrt, simplify
from sympy.physics.quantum import TensorProduct
from sympy.physics.quantum.gate import HadamardGate
from sympy.physics.quantum.represent import represent

from oracle import oracle

## Oracle examples

### Test 0 -- superposed query, constant functions

From *Introduction to Classical and Quantum Computing*, Exercise 7.3.

The query register is a weighted superposition:

$$|x\rangle = \frac{\sqrt{3}}{2}|0\rangle + \frac{1}{2}|1\rangle$$

The ancilla is prepared in the minus state:

$$|y\rangle = H|1\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}} = |{-}\rangle$$

We test two constant functions:
- $f_0(x) = 0$ for all $x$ -- the ancilla XOR leaves $|y\rangle$ unchanged
- $f_1(x) = 1$ for all $x$ -- the XOR flips $|y\rangle$, giving $-|{-}\rangle$

Expected results:

$$U_{f_0}\,|x\rangle|{-}\rangle
  = \frac{\sqrt{3}}{2\sqrt{2}}(|00\rangle - |01\rangle)
  + \frac{1}{2\sqrt{2}}(|10\rangle - |11\rangle)$$

$$U_{f_1}\,|x\rangle|{-}\rangle
  = \frac{\sqrt{3}}{2\sqrt{2}}(|01\rangle - |00\rangle)
  + \frac{1}{2\sqrt{2}}(|11\rangle - |10\rangle)
  = -U_{f_0}\,|x\rangle|{-}\rangle$$

The global $-1$ phase from $f_1$ is exactly the phase-kickback effect.

In [2]:
def test_0():
    """Exercise 7.3: superposed query with constant-zero and
    constant-one oracles; ancilla is the minus state."""
    x = sqrt(3) / 2 * Qubit(0) + 1 / 2 * Qubit(1)
    print(f"|x> = {x}")

    # ancilla: |-> = H|1> = (|0> - |1>) / sqrt(2)
    y = qapply(HadamardGate(0) * Qubit(1))
    print(f"|y> = {y}")

    xy = TensorProduct(x, y)
    xy = matrix_to_qubit(represent(xy))
    print(f"|xy> = {xy}")

    # constant-zero function: f(x) = 0 for all x
    # XOR with 0 leaves y unchanged -> output equals input
    def f0(x_bits, *args):
        return 0

    result = simplify(oracle(x, y, f0))
    print(f"f0 result: {result}")
    expected = simplify(
        sqrt(3) / (2 * sqrt(2)) * (Qubit("00") - Qubit("01"))
        + 1 / (2 * sqrt(2)) * (Qubit("10") - Qubit("11"))
    )
    assert simplify(result - expected) == 0

    # constant-one function: f(x) = 1 for all x
    # XOR with 1 flips y; with |-> ancilla this produces a global -1 phase
    def f1(x_bits, *args):
        return 1

    result = oracle(x, y, f1)
    print(f"f1 result: {result}")
    expected = simplify(
        sqrt(3) / (2 * sqrt(2)) * (Qubit("01") - Qubit("00"))
        + 1 / (2 * sqrt(2)) * (Qubit("11") - Qubit("10"))
    )
    assert simplify(result - expected) == 0


test_0()

|x> = sqrt(3)*|0>/2 + 0.5*|1>
|y> = sqrt(2)*|0>/2 - sqrt(2)*|1>/2
|xy> = sqrt(6)*|00>/4 - sqrt(6)*|01>/4 + 0.25*sqrt(2)*|10> - 0.25*sqrt(2)*|11>
f0 result: sqrt(2)*(sqrt(3)*|00> - sqrt(3)*|01> + |10> - 1.0*|11>)/4
f1 result: -sqrt(6)*|00>/4 + sqrt(6)*|01>/4 - 0.25*sqrt(2)*|10> + 0.25*sqrt(2)*|11>


### Test 1 -- phase-kickback with basis state query

The phase-kickback identity states:

$$U_f\,|x\rangle|{-}\rangle = (-1)^{f(x)}\,|x\rangle|{-}\rangle$$

We verify this for $|x\rangle = |0\rangle$ and both constant functions:

- $f_0$: $(-1)^0 = +1$ -- no phase change
- $f_1$: $(-1)^1 = -1$ -- global sign flip

The ancilla $|{-}\rangle$ is unchanged in both cases.

In [3]:
def test_1():
    """Verify phase-kickback: U_f|x>|-> = (-1)^f(x) |x>|->."""
    x = Qubit(0)
    print(f"|x> = {x}")

    # ancilla: |-> = H|1>
    y = qapply(HadamardGate(0) * Qubit(1))
    print(f"|y> = {y}")

    # f(x) = 0: phase is (-1)^0 = 1, output equals input
    def f0(x_bits, *args):
        return 0

    result = oracle(x, y, f0)
    print(f"f0 result: {result}")
    expected = matrix_to_qubit(represent(
        (-1) ** f0(x) * TensorProduct(x, y)
    ))
    assert expected == result

    # f(x) = 1: phase is (-1)^1 = -1, output is negated
    def f1(x_bits, *args):
        return 1

    result = oracle(x, y, f1)
    print(f"f1 result: {result}")
    expected = matrix_to_qubit(represent(
        (-1) ** f1(x) * TensorProduct(x, y)
    ))
    assert expected == result


test_1()

|x> = |0>
|y> = sqrt(2)*|0>/2 - sqrt(2)*|1>/2
f0 result: sqrt(2)*|00>/2 - sqrt(2)*|01>/2
f1 result: -sqrt(2)*|00>/2 + sqrt(2)*|01>/2


### Test 2 -- constant-zero function, $|x\rangle = |0\rangle$, $|y\rangle = |0\rangle$

Simplest case: both registers are basis states.

$$U_{f_0}\,|0\rangle|0\rangle = |0\rangle|0 \oplus 0\rangle = |0\rangle|0\rangle$$

The state is unchanged because $f_0(x) = 0$ and $0 \oplus 0 = 0$.

In [4]:
def test_2():
    """Constant-zero oracle on |0>|0>: output must be |0>|0>."""
    x = Qubit(0)
    y = Qubit(0)
    print(f"|x> = {x},  |y> = {y}")

    # f(x) = 0: y XOR 0 = y, so output ancilla equals input ancilla
    def f(x_bits, *args):
        return 0

    result = oracle(x, y, f)
    print(f"result: {result}")

    expected = matrix_to_qubit(represent(TensorProduct(x, Qubit(f(0)))))
    assert expected == result


test_2()

|x> = |0>,  |y> = |0>
result: |00>


### Test 3 -- constant-zero function, $|x\rangle = |1\rangle$, $|y\rangle = |0\rangle$

$$U_{f_0}\,|1\rangle|0\rangle = |1\rangle|0 \oplus 0\rangle = |1\rangle|0\rangle$$

The query register value does not matter for a constant function;
the ancilla is still unchanged.

In [5]:
def test_3():
    """Constant-zero oracle on |1>|0>: output must be |1>|0>."""
    x = Qubit(1)
    y = Qubit(0)
    print(f"|x> = {x},  |y> = {y}")

    # f(x) = 0 regardless of x
    def f(x_bits, *args):
        return 0

    result = oracle(x, y, f)
    print(f"result: {result}")

    expected = matrix_to_qubit(represent(TensorProduct(x, Qubit(f(1)))))
    assert expected == result


test_3()

|x> = |1>,  |y> = |0>
result: |10>


### Test 4 -- constant-one function, $|x\rangle = |0\rangle$, $|y\rangle = |0\rangle$

$$U_{f_1}\,|0\rangle|0\rangle = |0\rangle|0 \oplus 1\rangle = |0\rangle|1\rangle$$

The ancilla is flipped because $f_1(x) = 1$ and $0 \oplus 1 = 1$.

In [6]:
def test_4():
    """Constant-one oracle on |0>|0>: ancilla must flip to |1>."""
    x = Qubit(0)
    y = Qubit(0)
    print(f"|x> = {x},  |y> = {y}")

    # f(x) = 1: y XOR 1 flips the ancilla bit
    def f(x_bits, *args):
        return 1

    result = oracle(x, y, f)
    print(f"result: {result}")

    expected = matrix_to_qubit(represent(TensorProduct(x, Qubit(f(0)))))
    assert expected == result


test_4()

|x> = |0>,  |y> = |0>
result: |01>


### Test 5 -- constant-one function, $|x\rangle = |1\rangle$, $|y\rangle = |0\rangle$

$$U_{f_1}\,|1\rangle|0\rangle = |1\rangle|0 \oplus 1\rangle = |1\rangle|1\rangle$$

Again the ancilla is flipped, independent of the query value.

In [7]:
def test_5():
    """Constant-one oracle on |1>|0>: ancilla must flip to |1>."""
    x = Qubit(1)
    y = Qubit(0)
    print(f"|x> = {x},  |y> = {y}")

    # f(x) = 1 regardless of x; ancilla flips
    def f(x_bits, *args):
        return 1

    result = oracle(x, y, f)
    print(f"result: {result}")

    expected = matrix_to_qubit(represent(TensorProduct(x, Qubit(f(1)))))
    assert expected == result


test_5()

|x> = |1>,  |y> = |0>
result: |11>


### Test 6 -- unitarity: applying $U_f$ twice returns the original state

Because $U_f$ is its own inverse (self-adjoint when restricted to XOR),
applying it twice must recover the input:

$$U_f\bigl(U_f\,|x\rangle|y\rangle\bigr)
  = |x\rangle\,|y \oplus f(x) \oplus f(x)\rangle
  = |x\rangle|y\rangle$$

Numeric trace for $|x\rangle = |1\rangle$, $|y\rangle = |0\rangle$, $f(x)=1$:

1. First application:  $|1\rangle|0 \oplus 1\rangle = |1\rangle|1\rangle$
2. Second application: $|1\rangle|1 \oplus 1\rangle = |1\rangle|0\rangle$ -- back to start

In [8]:
def test_6():
    """Unitarity: applying oracle twice recovers the original state."""
    x = Qubit(1)
    y = Qubit(0)
    print(f"|x> = {x},  |y> = {y}")

    def f(x_bits, *args):
        return 1

    # first application: |1>|0> -> |1>|1>
    result = oracle(x, y, f)
    print(f"after first application: {result}")
    expected = matrix_to_qubit(represent(TensorProduct(x, Qubit(f(1)))))
    assert expected == result

    # second application: |1>|1> -> |1>|0> (back to original)
    result = oracle(x, Qubit(f(1)), f)
    print(f"after second application: {result}")
    expected = matrix_to_qubit(represent(TensorProduct(x, y)))
    assert expected == result


test_6()

|x> = |1>,  |y> = |0>
after first application: |11>
after second application: |10>


### Test 7 -- unitarity with phase-kickback ancilla

The same unitarity property holds even when the ancilla is $|{-}\rangle$.
With $|x\rangle = |0\rangle$ and $f(x) = 1$:

1. First application:  $U_f|0\rangle|{-}\rangle = (-1)^1|0\rangle|{-}\rangle = -|0\rangle|{-}\rangle$
2. Second application: $U_f(-|0\rangle)|{-}\rangle = -(-1)^1|0\rangle|{-}\rangle = |0\rangle|{-}\rangle$

The double application cancels the phase, restoring the original state.

In [9]:
def test_7():
    """Unitarity with |-> ancilla: two applications cancel the phase."""
    x = Qubit(0)
    print(f"|x> = {x}")

    # ancilla: |-> = H|1>
    y = qapply(HadamardGate(0) * Qubit(1))
    print(f"|y> = {y}")

    def f(x_bits, *args):
        return 1

    # first application: phase kickback gives -|0>|->.
    result = oracle(x, y, f)
    print(f"after first application: {result}")
    expected = matrix_to_qubit(represent(
        TensorProduct((-1) ** f(x) * x, y)
    ))
    assert expected == result

    # second application: the phased query -|0> goes back to |0>|->.
    result = oracle((-1) ** f(x) * x, y, f)
    print(f"after second application: {result}")
    expected = matrix_to_qubit(represent(TensorProduct(x, y)))
    assert expected == result


test_7()

|x> = |0>
|y> = sqrt(2)*|0>/2 - sqrt(2)*|1>/2
after first application: -sqrt(2)*|00>/2 + sqrt(2)*|01>/2
after second application: sqrt(2)*|00>/2 - sqrt(2)*|01>/2
